In [ ]:
import os
import logging
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from jax.experimental.ode import odeint
from lsh_qudit.matrices import hamiltonian, hamiltonian_op, nl_bounds
from lsh_qudit.agl import boundary_conditions

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
jax.config.update('jax_enable_x64', True)
logging.basicConfig(level=logging.INFO)

### Energy density function

In [ ]:
def energy_density(probs, shape):
    dim = np.prod(shape)
    offsets = np.cumprod((1,) + shape[:0:-1])[::-1]
    nvals = (np.arange(dim)[:, None] // offsets[None, :]) % np.array(shape)[None, :]
    nvals = np.flip(nvals, axis=1)
    ni = nvals[:, ::3]
    no = nvals[:, 1::3]
    nl = nvals[:, 2::3]
    site_energy_densities = (nl / 2. + nl * nl / 4. + (nl + 1.5) * no * (1 - ni) / 2.)
    extra_dims = tuple(range(probs.ndim - 1))
    expval = np.sum(probs[..., None] * np.expand_dims(site_energy_densities, extra_dims), axis=-2)
    return expval

### Set up the Hamiltonian operator and integrate the Schrodinger equation

In [ ]:
num_sites = 6
tmax = 5.
tpoints = np.linspace(0., tmax, 201)
mass_mu = 1.
interaction_x = 1.

# i(0), o(0), l(0), i(1), o(1), l(1), ...
# initial_state = [0, 1, 0] + [0, 1, 1] + [0, 0, 2] + [1, 0, 1]
initial_state = [0, 1, 0] + [0, 1, 1] + [0, 0, 2] + [1, 1, 2] + [1, 0, 1] + [1, 0, 0]

In [ ]:
# Function that computes H|psi>
hop = hamiltonian_op(num_sites, mass_mu, interaction_x)

# Time derivative of exp(-iHt)|psi(0)>
def dpsidt(state, t):
    return -1.j * hop(state)

shape = (3, 2, 2) * num_sites
psi0 = np.zeros(shape, dtype=np.complex128)
psi0[tuple(initial_state[::-1])] = 1.
psi0 = psi0.reshape(-1)

# Solve the Schrodinger equation
states_ode = odeint(dpsidt, psi0, tpoints)
probs = np.square(np.abs(states_ode))
# Integrate out the dof of the last site
probs = np.sum(probs.reshape(-1, np.prod(shape[:3]), np.prod(shape[3:])), axis=1)
# Compute the energy density of each site at each time
expval_ode = energy_density(probs, shape[3:])


In [ ]:
plt.plot(tpoints, expval_ode, label=[f'$r={r}$' for r in range(num_sites - 1)])
plt.legend()
plt.xlabel('t')
plt.ylabel(r'$\langle H_E(r) \rangle$')

### Matrix-based integration (validation)

In [ ]:
num_sites = 4
tmax = 5.
tpoints = np.linspace(0., tmax, 201)
mass_mu = 1.
interaction_x = 1.

# i(0), o(0), l(0), i(1), o(1), l(1), ...
initial_state = [0, 1, 0] + [0, 1, 1] + [0, 0, 2] + [1, 0, 1]

In [ ]:
hmat = hamiltonian(num_sites, mass_mu, interaction_x,
                   max_left_flux=0, max_right_flux=1, npmod=jnp)

def dpsidt(state, t, hmat):
    return -1.j * hmat @ state

conditions = boundary_conditions(num_sites, 0, 1)
shape = ()
for bc in conditions:
    shape = (nl_bounds(**bc)[1], 2, 2) + shape
dim = np.prod(shape)
offsets = np.cumprod((1,) + shape[:0:-1])[::-1]
idx = np.sum(offsets * np.array(initial_state[::-1]))
psi0 = np.zeros(dim, dtype=np.complex128)
psi0[idx] = 1.

# Solve the Schrodinger equation
states_ode = odeint(dpsidt, psi0, tpoints, hmat)
probs = np.square(np.abs(states_ode))
# Integrate the site 4 dimensions
probs = np.sum(probs.reshape(-1, np.prod(shape[:3]), np.prod(shape[3:])), axis=1)
expval_ode = energy_density(probs, shape[3:])


In [ ]:
plt.plot(expval_ode)